## Question 1
Converting PDFs to Markdown

In [2]:
!uv add 'markitdown[pdf]'


Resolved 158 packages in 6ms
Checked 141 packages in 26ms


In [3]:
import pandas as pd

books_csv_url = 'https://raw.githubusercontent.com/alexeygrigorev/ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv'
books_df = pd.read_csv(books_csv_url)
books_df


,title,book_url,pdf_url
0,Think Python 2e,https://greenteapress.com/wp/think-python-2e/,https://greenteapress.com/thinkpython2/thinkpy...
1,Think DSP,https://greenteapress.com/wp/think-dsp/,https://greenteapress.com/thinkdsp/thinkdsp.pdf
2,Think Complexity 2e,https://greenteapress.com/wp/think-complexity/,https://greenteapress.com/complexity2/thinkcom...
3,Think Java 2e,https://greenteapress.com/wp/think-java-2e/,https://greenteapress.com/thinkjava7/thinkjava...
4,Physical Modeling in MATLAB,https://greenteapress.com/wp/physical-modeling...,https://github.com/AllenDowney/PhysicalModelin...
5,Think OS,https://greenteapress.com/wp/think-os/,https://greenteapress.com/thinkos/thinkos.pdf
6,Think C++,https://greenteapress.com/wp/think-c/,https://raw.githubusercontent.com/tscheffl/Thi...


In [4]:
import requests
from pathlib import Path

books_dir = Path('books')
books_dir.mkdir(exist_ok=True)

for _, row in books_df.iterrows():
    pdf_response = requests.get(row['pdf_url'])
    filename = books_dir / f"{row['title']}.pdf"
    filename.write_bytes(pdf_response.content)
    print(f"Saved {filename}")


Saved books/Think Python 2e.pdf
Saved books/Think DSP.pdf
Saved books/Think Complexity 2e.pdf


KeyboardInterrupt: 

In [5]:
from markitdown import MarkItDown

books_text_dir = Path('books_text')
books_text_dir.mkdir(exist_ok=True)

md_converter = MarkItDown()

for pdf_path in books_dir.glob('*.pdf'):
    result = md_converter.convert(str(pdf_path))
    text_path = books_text_dir / f"{pdf_path.stem}.md"
    text_path.write_text(result.text_content)
    print(f"Converted {pdf_path.name} -> {text_path.name}")

Converted Think OS.pdf -> Think OS.md
Converted Think Java 2e.pdf -> Think Java 2e.md
Converted Physical Modeling in MATLAB.pdf -> Physical Modeling in MATLAB.md
Converted Think Python 2e.pdf -> Think Python 2e.md
Converted Think DSP.pdf -> Think DSP.md
Converted Think Complexity 2e.pdf -> Think Complexity 2e.md
Converted Think C++.pdf -> Think C++.md


In [6]:
think_python_path = books_text_dir / 'Think Python 2e.md'
line_count = len(think_python_path.read_text().splitlines())
line_count

9627

## Question 2
Chunking for RAG

In [7]:
# Read each markdown file, clean lines, build the doc dicts

from pathlib import Path

books_text_dir = Path('books_text')

book_docs = []
for md_path in books_text_dir.glob('*.md'):
    lines = md_path.read_text().split('\n')
    non_empty_lines = [line for line in lines if line.strip()]
    book_docs.append({
        'source': md_path.name,
        'content': non_empty_lines
    })

len(book_docs)

7

In [8]:
!uv add gitsource

Resolved 158 packages in 19ms
Checked 141 packages in 82ms


In [9]:
from gitsource import chunk_documents

book_chunks = chunk_documents(book_docs, size=100, step=50)
len(book_chunks)


995

In [13]:
think_python_chunks = [c for c in book_chunks if c['source'] == 'thinkpython2.md']
len(think_python_chunks)

0